> **Solución.** Challenge de ML (pipeline de clasificación de vinos) con los `TODO` completados y las preguntas respondidas. Los tres notebooks se ejecutan **en orden** (1 → 2 → 3) y comparten artefactos en `data/`, `artifacts/` y `reports/`.
>
> Nicolás Rodríguez

# Challenge 1: datos y análisis exploratorio

Tu primera responsabilidad es preparar una fuente de datos confiable para el proyecto de clasificación de vinos.

**Entrada:** `data/raw/wine.csv`  
**Salidas:** `data/processed/train.csv`, `data/processed/test.csv` y `artifacts/data_contract.json`

> El conjunto de test se crea aquí, pero no debe explorarse ni utilizarse después del split.

## Objetivos

- cargar y validar datos externos;
- identificar problemas básicos de calidad;
- crear un split reproducible y estratificado;
- realizar EDA únicamente sobre entrenamiento;
- guardar datos y metadatos para la siguiente etapa.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga y profiling

In [3]:
DATA_PATH = RAW_DATA_DIR / "wine.csv"
df = pd.read_csv(DATA_PATH)
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [4]:
print("Forma:", df.shape)
print("\nTipos:\n", df.dtypes)
print("\nNulos por columna:\n", df.isna().sum())
print("\nFilas duplicadas:", df.duplicated().sum())
print("\nDistribución de target:\n", df["target"].value_counts().sort_index())
print("\nProporción de target:\n", df["target"].value_counts(normalize=True).sort_index().round(3))

Forma: (178, 14)

Tipos:
 alcohol                         float64
malic_acid                      float64
ash                             float64
alcalinity_of_ash               float64
magnesium                       float64
total_phenols                   float64
flavanoids                      float64
nonflavanoid_phenols            float64
proanthocyanins                 float64
color_intensity                 float64
hue                             float64
od280/od315_of_diluted_wines    float64
proline                         float64
target                            int64
dtype: object

Nulos por columna:
 alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280

**Conclusión de calidad:**

**Conclusión de calidad.** El dataset tiene 178 filas y 14 columnas (13 numéricas + `target`), **sin nulos y sin duplicados**. El `target` tiene 3 clases razonablemente balanceadas (aprox. 33 % / 40 % / 27 %). Decisión: no hace falta limpieza; solo se estandariza dentro del pipeline para los modelos sensibles a la escala.

## 2. Split train/test

In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],
    random_state=RANDOM_STATE,
)

In [6]:
# Verificación del split
assert len(train_df) + len(test_df) == len(df)
assert set(train_df.index).isdisjoint(set(test_df.index))
print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (142, 14)
Test: (36, 14)


## 3. EDA sobre entrenamiento

Incluye:

- distribución de clases;
- distribuciones o boxplots de variables relevantes;
- correlaciones;
- tres hallazgos que puedan influir en el modelado.

In [7]:
features = [c for c in train_df.columns if c != "target"]

# 1) distribución de clases en entrenamiento
train_df["target"].value_counts().sort_index().plot.bar(title="Distribución de clases (train)")
plt.xlabel("target"); plt.ylabel("conteo"); plt.show()

# 2) boxplots de algunas variables discriminantes por clase
for col in ["flavanoids", "color_intensity", "proline"]:
    sns.boxplot(data=train_df, x="target", y=col)
    plt.title(f"{col} por clase"); plt.show()

# 3) mapa de correlaciones
corr = train_df[features].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True)
plt.title("Correlaciones entre variables (train)"); plt.tight_layout(); plt.show()

print("Rangos muy distintos entre variables (p.ej. proline ~ cientos vs hue ~ 1):",
      train_df[features].describe().loc[["min", "max"]].T)

Rangos muy distintos entre variables (p.ej. proline ~ cientos vs hue ~ 1):                                  min      max
alcohol                        11.03    14.83
malic_acid                      0.74     5.80
ash                             1.36     3.22
alcalinity_of_ash              10.60    30.00
magnesium                      70.00   162.00
total_phenols                   0.98     3.88
flavanoids                      0.34     3.74
nonflavanoid_phenols            0.13     0.63
proanthocyanins                 0.42     3.58
color_intensity                 1.28    13.00
hue                             0.48     1.71
od280/od315_of_diluted_wines    1.27     3.92
proline                       278.00  1515.00


/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_34236/2323421568.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("target"); plt.ylabel("conteo"); plt.show()
/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_34236/2323421568.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title(f"{col} por clase"); plt.show()
/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_34236/2323421568.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title(f"{col} por clase"); plt.show()
/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_34236/2323421568.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title(f"{col} por clase"); plt.show()
/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_34236/2323421568.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Correlaciones 

**Hallazgos:**

1.
2.
3.

**Hallazgos (solo sobre train).**
1. Las escalas son muy dispares: `proline` llega a ~1500 mientras `hue` ronda 1 → hay que estandarizar para KNN y regresión logística.
2. `flavanoids`, `od280/od315` y `proline` separan visualmente bien las clases en los boxplots.
3. Hay correlaciones fuertes entre fenoles (`total_phenols`, `flavanoids`, `od280/od315`) → posible redundancia, pero con 13 features y 178 filas no es crítico.

## 4. Persistencia de datos y contrato

In [8]:
train_out = train_df.reset_index(drop=True)
test_out = test_df.reset_index(drop=True)
train_out.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_out.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

data_contract = {
    "target": "target",
    "features": [c for c in df.columns if c != "target"],
    "random_state": RANDOM_STATE,
    "test_size": 0.2,
}
(ARTIFACTS_DIR / "data_contract.json").write_text(
    json.dumps(data_contract, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(data_contract, indent=2, ensure_ascii=False))

{
  "target": "target",
  "features": [
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280/od315_of_diluted_wines",
    "proline"
  ],
  "random_state": 42,
  "test_size": 0.2
}


In [9]:
# Verificación de entrega
assert (PROCESSED_DATA_DIR / "train.csv").exists()
assert (PROCESSED_DATA_DIR / "test.csv").exists()
assert (ARTIFACTS_DIR / "data_contract.json").exists()
print("Challenge 1 completado.")

Challenge 1 completado.


## Checklist

- [ ] Cargué el CSV desde `data/raw`.
- [ ] Revisé calidad y distribución de la clase.
- [ ] Realicé un split estratificado.
- [ ] No exploré el conjunto de test.
- [ ] Guardé los tres artefactos requeridos.